# Heat capacity of LJ

864 Lennard-Jones atoms in 3D (6 × 6 × 6 fcc cells). After a short equilibration the heat-bath temperature is raised slowly from `Tstart` to `Tstop`. The heat capacity per atom is the slope of the total energy per atom against the temperature:

$$\frac{C_V^*}{N} = \frac{\mathrm{d}(E^*/N)}{\mathrm{d}T^*}$$

Change `variable rho` in _heat-capacity-lj.in_ to run the dilute gas (0.02), the liquid (0.85) and the solid (1.1). **Predict the ideal-gas and harmonic-solid limits.** Kinetic equipartition contributes 3/2 to $C_V/(Nk)$ in all three cases; it does not fix the liquid’s potential-energy contribution.

Here $C_V^*/N = C_V/(Nk)$. A straight-line fit over the finite heating ramp estimates an average slope, rather than the derivative at one temperature. The default ramp is 0.7–1.8. To test the harmonic-solid limit shown in the slides, keep `rho = 1.1` and change `Tstart` to 0.1 and `Tstop` to 0.3.

In [ ]:
# lammps-logfile is served from Atomify's own package index (no network needed).
%pip install -q lammps-logfile pandas

In [ ]:
import glob, os, time
import numpy as np
import lammps_logfile
import matplotlib.pyplot as plt

# Atomify stores every run of this project in runs/<run-name>/ next to this
# notebook (input snapshot, log.lammps, dumps). Pick the newest run here;
# use logs[0], logs[1], ... to look at older ones.
logs = sorted(glob.glob("runs/*/log.lammps"))
if not logs:
    raise RuntimeError("No runs yet: press Run in Atomify, wait for it to finish, then re-run this cell.")
print("Runs found:", *logs, sep="\n  ")

for attempt in range(5):
    try:
        log = lammps_logfile.File(logs[-1])
        break
    except FileNotFoundError:
        # Atomify may still be copying the finished run into the project.
        time.sleep(1)
        os.listdir(os.path.dirname(logs[-1]))
else:
    raise RuntimeError(f"{logs[-1]} is not readable yet: wait for the run to finish, then re-run this cell.")
print("Log keywords:", log.get_keywords())

Total energy per atom against temperature during the ramp (run 2 of the input file), with a straight-line fit:

In [ ]:
T = log.get("Temp",   run_num=1)     # the ramp is the second run in the input file
E = log.get("TotEng", run_num=1)     # per atom in LJ units
C, E0 = np.polyfit(T, E, 1)

plt.figure(figsize=(7, 4))
plt.plot(T, E, ".", ms=3, alpha=.5, label="simulation")
plt.plot([T.min(), T.max()], [C * T.min() + E0, C * T.max() + E0], "k-", label=f"fit: slope C*/N = {C:.2f}")
plt.xlabel("T*"); plt.ylabel("E*/N"); plt.legend(); plt.show()
print(f"heat capacity per atom C*/N = {C:.3f}")

All runs of this project in one plot:

In [ ]:
for path in logs:
    run = lammps_logfile.File(path)
    TT, EE = run.get("Temp", run_num=1), run.get("TotEng", run_num=1)
    plt.plot(TT, EE, ".", ms=2, label=f'{path.split("/")[1]}: C*/N = {np.polyfit(TT, EE, 1)[0]:.2f}')
plt.xlabel("T*"); plt.ylabel("E*/N"); plt.legend(); plt.show()

Note down:
* Gas: how many quadratic terms does each atom's energy have? What slope does equipartition predict, and what did you get?
* Solid: in the harmonic approximation, how many quadratic terms now (kinetic *and* potential)? Compare the fitted slope with 3 (Dulong–Petit); does the colder ramp approach this limit?
* Liquid: why does kinetic equipartition still hold, while the total heat capacity has no universal value?
* Does the slope change along the ramp for any of the densities — and what would that mean?